# 1. Setup & Installation
Install required dependencies and import libraries.

In [ ]:
%pip install -q --no-cache-dir --force-reinstall -U "google-genai>=1.16.0"
%pip install -q gspread-dataframe gspread_pandas google-cloud-aiplatform rouge_score plotly jsonlines anthropic
%pip install -q --upgrade --force-reinstall numpy sentence-transformers scikit-learn

In [ ]:
import os
import time
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from tenacity import retry, wait_random_exponential, stop_after_attempt

import gspread
from google.auth import default
from google.colab import auth
from gspread_dataframe import set_with_dataframe

from google import genai
from google.genai import types
from openai import OpenAI
from huggingface_hub import InferenceClient
import anthropic

# Authenticate for Google Sheets access
auth.authenticate_user()

# 2. Core Libraries & API Initialization
Define the APIs and the resilient model-calling functions.

In [ ]:
# === API Keys & Models Configuration ===
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY" # @param {type:"string"}
OPENAI_MODEL = "o4-mini-2025-04-16" # @param {type:"string"}

HF_TOKEN = "YOUR_HF_TOKEN" # @param {type:"string"}
HF_MODEL = "meta-llama/Llama-4-Scout-17B-16E-Instruct" # @param {type:"string"}

GEMINI_API_KEY = "YOUR_GEMINI_API_KEY" # @param {type:"string"}
GEMINI_MODEL = "gemini-2.5-flash" # @param {type:"string"}

ANTHROPIC_API_KEY = "YOUR_ANTHROPIC_API_KEY" # @param {type:"string"}
ANTHROPIC_MODEL = "claude-haiku-4-5-20251001" # @param {type:"string"}

# === API Initializations ===
client_openai = OpenAI(api_key=OPENAI_API_KEY)
hf_client = InferenceClient(token=HF_TOKEN)
gemini_client = genai.Client(api_key=GEMINI_API_KEY)
client_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# === Tenacious Model Calls ===
@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(5))
def call_gpt(prompt):
    completion = client_openai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return completion.choices[0].message.content

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(5))
def call_llama(prompt):
    response = hf_client.chat_completion(
        model=HF_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.9,
        max_tokens=1000,
    )
    return response.choices[0].message.content

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(5))
def call_gemini(prompt):
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            safety_settings=[
                types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
            ]
        )
    )
    return response.text

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(5))
def call_claude(prompt):
    message = client_anthropic.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=1000,
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

# 3. Data Loading
Parameterize the data ingestion source.

In [ ]:
input_sheet_url = 'https://docs.google.com/spreadsheets/d/1DkRJYvpoe8K8dTZfRgn-O_cywnTiTxafdJhk0d-fapA/edit?gid=809673090#gid=809673090' # @param {type:"string"}
input_tab_name = 'human' # @param {type:"string"}

def get_sheet_data(sheet_url, tab_name):
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.open_by_url(sheet_url)
    worksheet = sh.worksheet(tab_name)
    data = worksheet.get_all_values()
    df = pd.DataFrame(data[1:], columns=data[0])
    return df

print(f"Loading data from tab: '{input_tab_name}'...")
df_all = get_sheet_data(input_sheet_url, input_tab_name)

# Ensure 'prompts' column exists. Modify this if your source column is named differently (e.g., 'query')
if 'query' in df_all.columns and 'prompts' not in df_all.columns:
    df_all = df_all.rename(columns={'query': 'prompts'})

display(df_all.head())

# 4. Execution
Run the inference logic. A Dry Run mode is included for fast testing.

In [ ]:
dry_run = True # @param {type:"boolean"}
dry_run_rows = 5 # @param {type:"integer"}

if dry_run:
    print(f"[DRY RUN] Executing on the first {dry_run_rows} rows only...")
    df_to_process = df_all.head(dry_run_rows).copy()
else:
    print(f"[PRODUCTION] Executing on all {len(df_all)} rows...")
    df_to_process = df_all.copy()

models_config = [
    (OPENAI_MODEL, call_gpt),
    (HF_MODEL, call_llama),
    (GEMINI_MODEL, call_gemini),
    (ANTHROPIC_MODEL, call_claude)
]

def process_prompt_safe(prompt, api_func):
    try:
        return api_func(prompt)
    except Exception as e:
        return f"ERROR: {str(e)}"

# Process inferences sequentially by model, but concurrently across prompts to remain clean in-memory
for model_name, api_func in models_config:
    print(f"\nEvaluating: {model_name}")
    results = []

    # Using ThreadPoolExecutor.map keeps the output order identical to the input list
    prompts_list = df_to_process['prompts'].tolist()
    with ThreadPoolExecutor(max_workers=5) as executor:
        results = list(tqdm(
            executor.map(lambda p: process_prompt_safe(p, api_func), prompts_list),
            total=len(prompts_list),
            desc=model_name
        ))

    df_to_process[model_name] = results

display(df_to_process.head())

# 5. Export Results
Save the final combined inferences back to Google Sheets.

In [ ]:
destination_sheet_url = 'https://docs.google.com/spreadsheets/d/14woRrQ4wOwffENgNZshr2E_7MH8iOY2wW5IzjCsSbOQ/edit?resourcekey=0-W8XcA7ryGKrRyHJo-owLeA&gid=178802188#gid=178802188' # @param {type:"string"}
destination_tab_name = 'Clean_Export_Results' # @param {type:"string"}

def export_to_sheets(df, sheet_url, tab_name, chunk_size=500):
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.open_by_url(sheet_url)

    try:
        worksheet = sh.worksheet(tab_name)
        worksheet.clear()
    except gspread.exceptions.WorksheetNotFound:
        worksheet = sh.add_worksheet(title=tab_name, rows=str(len(df)+1), cols=str(len(df.columns)))

    clean_df = df.fillna('').astype(str)
    for col in clean_df.columns:
        clean_df[col] = clean_df[col].str[:49000] # Cap limit for Sheets API

    print(f"Starting export of {len(clean_df)} rows...")
    for i in range(0, len(clean_df), chunk_size):
        chunk = clean_df.iloc[i:i+chunk_size]
        start_row = 1 if i == 0 else i + 2
        set_with_dataframe(worksheet, chunk, row=start_row, col=1, include_column_header=(i == 0))
        time.sleep(2)

    print(f"Successfully exported {len(clean_df)} rows to tab '{tab_name}'.")

if destination_sheet_url:
    export_to_sheets(df_to_process, destination_sheet_url, destination_tab_name)
else:
    print("No destination URL provided. Export skipped.")